# Ingestão SIA - Base AM (PySUS)

Este notebook deve ser executado no WSL.

Ele baixa os arquivos do grupo `AM` apenas de `2025` para `data/raw`.

Também baixa o **DTB IBGE 2024** (Divisão Territorial Brasileira) e gera `data/reference/ibge_dtb_2024/municipios_ibge_dtb.parquet`, usado no notebook de consultas para exibir nomes de municípios (código DATASUS de 6 dígitos alinhado ao código completo de 7 dígitos do IBGE).

In [ ]:
from pathlib import Path
from pysus import SIA

base_dir = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = base_dir / "data" / "raw"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
sia = SIA().load()
files = sia.get_files("AM", year=2025)

In [ ]:
downloaded = sia.download(files, local_dir=str(output_dir))

downloaded

---

In [ ]:
import urllib.request
import zipfile

import pandas as pd

IBGE_DTB_URL = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
    "estrutura_territorial/divisao_territorial/2024/DTB_2024.zip"
)
ref_dir = base_dir / "data" / "reference" / "ibge_dtb_2024"
ref_dir.mkdir(parents=True, exist_ok=True)

zip_path = ref_dir / "DTB_2024.zip"
if not zip_path.exists():
    print("Baixando DTB IBGE 2024...")
    urllib.request.urlretrieve(IBGE_DTB_URL, zip_path)

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(ref_dir)

mun_ods = ref_dir / "RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods"
mun_parquet = ref_dir / "municipios_ibge_dtb.parquet"

df_mun = pd.read_excel(mun_ods, engine="odf", header=6)
cols = list(df_mun.columns)
cod_col = next(c for c in cols if "Completo" in str(c))
nome_col = next(c for c in cols if str(c).startswith("Nome_Munic"))

cd_ibge = (
    df_mun[cod_col]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(7)
)
out = pd.DataFrame(
    {"cd_mun_ibge": cd_ibge, "nm_municipio": df_mun[nome_col].astype(str)}
)
out["cd_mun_datasus"] = out["cd_mun_ibge"].str.slice(0, 6)
out = out.drop_duplicates(subset=["cd_mun_datasus"], keep="first")
out.to_parquet(mun_parquet, index=False)
print(f"Municipios IBGE gravados em {mun_parquet} ({len(out)} linhas).")